[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/langchain-ai/langchain-academy/blob/main/module-0/basics.ipynb) [![Open in LangChain Academy](https://cdn.prod.website-files.com/65b8cd72835ceeacd4449a53/66e9eba12c7b7688aa3dbb5e_LCA-badge-green.svg)](https://academy.langchain.com/courses/take/intro-to-langgraph/lessons/56295530-getting-set-up-video-guide)

# LangChain Academy

Welcome to LangChain Academy!

## Context

At LangChain, we aim to make it easy to build LLM applications. One type of LLM application you can build is an agent. There’s a lot of excitement around building agents because they can automate a wide range of tasks that were previously impossible.

In practice though, it is incredibly difficult to build systems that reliably execute on these tasks. As we’ve worked with our users to put agents into production, we’ve learned that more control is often necessary. You might need an agent to always call a specific tool first or use different prompts based on its state.

To tackle this problem, we’ve built [LangGraph](https://docs.langchain.com/oss/python/langgraph/overview) — a framework for building agent and multi-agent applications. Separate from the LangChain package, LangGraph’s core design philosophy is to help developers add better precision and control into agent workflows, suitable for the complexity of real-world systems.

## Course Structure

The course is structured as a set of modules, with each module focused on a particular theme related to LangGraph. You will see a folder for each module, which contains a series of notebooks. A video will accompany each notebook to help walk through the concepts, but the notebooks are also stand-alone, meaning that they contain explanations and can be viewed independently of the videos. Each module folder also contains a `studio` folder, which contains a set of graphs that can be loaded into [LangSmith Studio](https://docs.langchain.com/langsmith/quick-start-studio), our IDE for building LangGraph applications.

## Setup

Before you begin, please follow the instructions in the `README` to create an environment and install dependencies.

## Chat models

In this course, we'll use Chat Models, which take a sequence of messages as input and return messages as output. LangChain supports many models via [third-party integrations](https://docs.langchain.com/oss/python/integrations/chat). By default, the course will use  [ChatOpenAI](https://docs.langchain.com/oss/python/integrations/chat/openai) because it is both popular and performant. As noted, please ensure that you have an `OPENAI_API_KEY`.

Let's check that your `OPENAI_API_KEY` is set and, if not, you will be asked to enter it.

In [11]:
%%capture --no-stderr
%pip install --quiet -U langchain_openai langchain_core langchain_community langchain-tavily

In [16]:
import os, getpass
from google.colab import userdata


def _set_env(var: str):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"{var}: ")

os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')



[Here](https://docs.langchain.com/oss/python/langchain/models) is a useful how-to for all the things that you can do with chat models, but we'll show a few highlights below. If you've run `pip install -r requirements.txt` as noted in the README, then you've installed the `langchain-openai` package. With this, we can instantiate our `ChatOpenAI` model object. You can see pricing for various models [here](https://openai.com/api/pricing/). The notebooks will default to `gpt-4o` because it offers a good balance of quality, price, and speed, but you can also opt for the lower-priced `gpt-3.5` series or more recent models.

There are [a few standard parameters](https://docs.langchain.com/oss/python/langchain/models#parameters) that we can set with chat models. Two of the most common are:

* `model`: the name of the model
* `temperature`: the sampling temperature

`Temperature` controls the randomness or creativity of the model's output where low temperature (close to 0) is more deterministic and focused outputs. This is good for tasks requiring accuracy or factual responses. High temperature (close to 1) is good for creative tasks or generating varied responses.

In [19]:
# 替换你的这部分代码
from langchain_openai import ChatOpenAI

# 配置密钥（建议使用环境变量，不要明文写在代码里）
# os.environ["ARK_API_KEY"] = "ak-xxxxxxxxxxxxxxxx"
os.environ["VOLC_BASE_URL"] = "https://ark.cn-beijing.volces.com/api/v3"

# 初始化火山引擎客户端
gpt4o_chat = ChatOpenAI(
    model="ep-20260406203409-dcqmk",
    temperature=0,
    base_url=os.getenv("VOLC_BASE_URL"),
    api_key=os.getenv("OPENAI_API_KEY"),
    streaming=True
)
gpt35_chat = ChatOpenAI(
    model="ep-20260406203500-qwgm2",
    temperature=0,
    base_url=os.getenv("VOLC_BASE_URL"),
    api_key=os.getenv("OPENAI_API_KEY"),
    streaming=True
)

# 后续的 .invoke() 和 .stream() 方法无需任何修改即可运行

Chat models in LangChain have a number of [default methods](https://reference.langchain.com/python/langchain_core/runnables). For the most part, we'll be using:

* [stream](https://docs.langchain.com/oss/python/langchain/models#stream): stream back chunks of the response
* [invoke](https://docs.langchain.com/oss/python/langchain/models#invoke): call the chain on an input

And, as mentioned, chat models take [messages](https://docs.langchain.com/oss/python/langchain/messages) as input. Messages have a role (that describes who is saying the message) and a content property. We'll be talking a lot more about this later, but here let's just show the basics.

In [21]:
from langchain_core.messages import HumanMessage

# Create a message
msg = HumanMessage(content="Hello world", name="Lance")

# Message list
messages = [msg]

# Invoke the model with a list of messages
gpt4o_chat.invoke(messages)

AIMessage(content='Hello Lance！很高兴认识你呀😊\n看你发了「Hello world」，是刚入门编程在写第一个测试代码，还是单纯想和世界打个招呼呀？有什么想聊的都可以随时和我说哦~', additional_kwargs={}, response_metadata={'finish_reason': 'stop', 'model_name': 'doubao-seed-2-0-pro-260215', 'service_tier': 'default', 'model_provider': 'openai'}, id='lc_run--019d679c-87a8-7ef0-920d-252c7f16eafa', tool_calls=[], invalid_tool_calls=[])

We get an `AIMessage` response. Also, note that we can just invoke a chat model with a string. When a string is passed in as input, it is converted to a `HumanMessage` and then passed to the underlying model.


In [ ]:
gpt4o_chat.invoke("hello world")

AIMessage(content='Hello! How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 9, 'prompt_tokens': 9, 'total_tokens': 18, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_cbf1785567', 'id': 'chatcmpl-CSWGCXlVYTEWoHAFm3GXIIgUO1gCb', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--43b070a6-7676-4aa1-984c-c0cd43d45c1e-0', usage_metadata={'input_tokens': 9, 'output_tokens': 9, 'total_tokens': 18, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [22]:
gpt35_chat.invoke("hello world")

AIMessage(content='Hello there! 👋\n\n作为编程世界里最经典的第一行输出，`hello world`可是所有开发者踏入编程大门的标志性起点呀～不管你是想聊编程相关的问题，还是有其他想分享、咨询的内容，都可以随时告诉我哦😉', additional_kwargs={}, response_metadata={'finish_reason': 'stop', 'model_name': 'doubao-seed-2-0-lite-260215', 'service_tier': 'default', 'model_provider': 'openai'}, id='lc_run--019d679c-e0f7-77b1-86a0-6f4df1d722c9', tool_calls=[], invalid_tool_calls=[])

The interface is consistent across all chat models and models are typically initialized once at the start up each notebooks.

So, you can easily switch between models without changing the downstream code if you have strong preference for another provider.


## Search Tools

You'll also see [Tavily](https://tavily.com/) in the README, which is a search engine optimized for LLMs and RAG, aimed at efficient, quick, and persistent search results. As mentioned, it's easy to sign up and offers a generous free tier. Some lessons (in Module 4) will use Tavily by default but, of course, other search tools can be used if you want to modify the code for yourself.

In [23]:
_set_env("TAVILY_API_KEY")

TAVILY_API_KEY: ··········


In [28]:
from langchain_tavily import TavilySearch  # updated at 1.0

tavily_search = TavilySearch(max_results=3)

data = tavily_search.invoke({"query": "什么是峨眉山"})
search_docs = data.get("results", data)

In [29]:
search_docs

[{'url': 'https://baike.baidu.com/item/%E5%B3%A8%E7%9C%89%E5%B1%B1/2676',
  'title': '峨眉山_百度百科',
  'content': '# 峨眉山. 峨眉山（Mount Emei），也作“峨嵋山” [22]属邛崃山脉支脉，地处中国四川盆地的西南边缘，介于北纬29°16′—29°43′，东经103°10′—103°37′之间，自峨眉平原拔地而起，山体南北延伸，绵延105千米。主要山峰为大峨山、二峨山、三峨山、四峨山，其中大峨山即为峨嵋山风景名胜区，面积为154平方千米，主峰金顶，最高峰万佛顶海拔3099米。 [1] [12] [23-24]. 因喜马拉雅运动，峨眉山主体沿断层强烈抬升，形成如今之峨眉山。 [12]峨眉山主体的地质基础为南北向短背斜，地貌按塑造地貌方式，可分为侵蚀地貌（峨眉山区）和堆积地貌（峨眉扇状冲洪积平原）；按成因可分为构造地貌、流水地貌、岩溶地貌和冰川地貌等。 [12]峨眉平原至万年寺以下低山丘陵区，主要是紫色土、黄泥土。 [1]峨眉山受季风环流影响2000米以上地区约有半年时间为冰雪覆盖，没有四季之分，只有冬春之别。 [12]. 峨眉山文化底蕴深厚。佛教、道教、武术、山茶文化在峨眉山蓬勃发展，多位历史名人在此留下诗篇；1996年，峨眉山绝大部分文物被联合国教科文组织列入《世界文化遗产》名录。 [6] [12] [15] [18]. ## 目录. ## 位置境域. 峨眉山地处中国四川盆地的西南边缘，介于北纬29°16′—29°43′，东经103°10′—103°37′之间，为邛崃山南段余脉，自峨眉平原拔地而起，山体南北延伸，绵延105千米，面积约110平方千米。 [1]. ## 历史成因. 震旦纪中后期至奥陶纪初期（距今7—5亿年左右），海水向中国西部、南部淹没而来，峨眉山区第二次沦为沧海，峨眉山区地壳缓慢沉降。. 早二叠纪时期（距今约2.7亿年），中国南方发生了地质史上最广泛的海浸，峨眉山区第三次沦为海底。. 始新世末期（距今约3000万年左右），在喜马拉雅运动作用下峨眉山山体沿着峨眉山大断层的断裂面迅速地抬升，形成峨眉山背斜，即峨眉山主体。. 喜马拉雅运动后期（距今约300万年左右），峨眉山主体沿断层强烈抬升，形成如今之峨眉山。 [12]. ## 地理特征. 